# Mission-Critical Access Gatekeeper - Interactive Demo

Hands-on notebook for face verification and emotion-risk policy gating.

It demonstrates why emotion analysis matters in high-impact systems: an authorized person may still be under coercion or severe distress. The flow uses multi-frame identity consensus, emotion batch voting, and a strict 2/2 pass rule before access is granted.

In [ ]:
import asyncio

import ipywidgets as widgets
from IPython.display import display

from deepface_security_framework.config.schema import AppConfig
from deepface_security_framework.gateway import SecurityGateway

In [ ]:
reference_path = widgets.Text(description='Reference', placeholder='C:/path/user.jpg')
admin_pool = widgets.Text(description='AdminPool', placeholder='C:/path/admins')

blocked = widgets.SelectMultiple(
    options=['angry', 'fear', 'disgust', 'sad', 'surprise', 'neutral', 'happy'],
    value=('angry', 'fear'),
    description='Blocked',
)
threshold = widgets.FloatSlider(value=0.8, min=0.1, max=1.0, step=0.01, description='EmotionThr')

identity_frames = widgets.IntSlider(value=5, min=1, max=12, step=1, description='IdFrames')
min_matches = widgets.IntSlider(value=2, min=1, max=12, step=1, description='IdMin')
identity_threshold = widgets.FloatSlider(value=0.9, min=0.3, max=1.2, step=0.01, description='IdThr')
emotion_frames = widgets.IntSlider(value=3, min=1, max=10, step=1, description='EmoFrames')
emotion_batches = widgets.IntSlider(value=2, min=1, max=8, step=1, description='EmoBatches')
show_window = widgets.Checkbox(value=True, description='Show camera window')

resource_name = widgets.Text(value='mission_critical_gateway', description='Resource')
run_btn = widgets.Button(description='Run Authorization', button_style='primary')
output = widgets.Output()

display(
    reference_path,
    admin_pool,
    blocked,
    threshold,
    identity_frames,
    min_matches,
    identity_threshold,
    emotion_frames,
    emotion_batches,
    show_window,
    resource_name,
    run_btn,
    output,
)

In [ ]:
async def run_secure():
    config = AppConfig()

    # Identity source: fill one source (reference file or admin pool folder).
    config.identity.reference_image_path = reference_path.value.strip() or None
    config.identity.admin_faces_dir = admin_pool.value.strip() or None

    # Identity consensus controls.
    config.identity.frames_per_check = int(identity_frames.value)
    config.identity.min_matches_required = int(min_matches.value)
    config.identity.distance_threshold = float(identity_threshold.value)

    # Emotion policy controls.
    config.emotion.blocked_emotions = list(blocked.value)
    config.emotion.threshold = float(threshold.value)
    config.emotion.frames_per_batch = int(emotion_frames.value)
    config.emotion.max_batches = int(emotion_batches.value)
    config.emotion.show_camera_window = bool(show_window.value)

    config.resource.resource_name = resource_name.value.strip() or config.resource.resource_name

    gateway = SecurityGateway.build(config)
    passed, response = await gateway.authorize_access()
    return passed, response


def on_click(_):
    with output:
        output.clear_output()
        if not (reference_path.value.strip() or admin_pool.value.strip()):
            print('Please provide either a reference image path or an admin pool directory.')
            return
        try:
            passed, response = asyncio.run(run_secure())
            if passed:
                print('ACCESS GRANTED')
                print(response)
            else:
                print('ACCESS DENIED')
                print(response)
        except Exception as exc:
            print(f'Execution failed: {exc}')


run_btn.on_click(on_click)